In [1]:
import pandas as pd
import matplotlib.pyplot as plt


# 1) Abrir o CSV (separador ';', decimal ',')

df = pd.read_csv(
    "Tabela4.csv",
    sep=";",
    decimal=",",
    encoding="latin1",
    skiprows=1,
)

print(df.head())
print(df.shape)


# 2) Limpar colunas vazias

df = df.dropna(axis=1, how="all")
print(df.isna().sum())


# 3) Ordenar os estados por maior IDH em 2024

df_ordenado = df.sort_values("2024", ascending=False)
print(df_ordenado[["Estado", "2024"]])


# 4) Qual estado teve a maior melhora de IDH entre 2024 e 1991?

df["melhora"] = df["2024"] - df["1991"]
maior_melhora = df.sort_values("melhora", ascending=False).iloc[0]
print(f"Maior melhora: {maior_melhora['Estado']} (+{maior_melhora['melhora']:.3f})")



# 5) Existe algum estado em que o IDH piorou?

pioraram = df[df["melhora"] < 0]
if pioraram.empty:
    print("Nenhum estado teve queda de IDH entre 1991 e 2024.")
else:
    print(pioraram[["Estado", "1991", "2024", "melhora"]])



# 5b) Formato largo -> longo (melt)

anos = [c for c in df.columns if str(c).isdigit()]
print(anos)

id_vars = [c for c in df.columns if c not in anos]
df_longo = df.melt(
    id_vars=id_vars,
    value_vars=anos,
    var_name="Ano",
    value_name="IDH",
)
df_longo["Ano"] = df_longo["Ano"].astype(int)
df_longo["IDH"] = pd.to_numeric(df_longo["IDH"], errors="coerce")
print(df_longo.head())


# 6) Plotar apenas Minas Gerais

mg = df_longo[df_longo["Sigla"] == "MG"].sort_values("Ano")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(mg["Ano"], mg["IDH"], marker="o", linewidth=2)
ax.set_title("Evolução do IDH — Minas Gerais (1991–2024)")
ax.set_xlabel("Ano")
ax.set_ylabel("IDH")
ax.set_ylim(0.3, 0.9)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()


# 7) Plotar a evolução do IDH de cada estado

fig, ax = plt.subplots(figsize=(12, 7))

for sigla, grupo in df_longo.groupby("Sigla"):
    grupo = grupo.sort_values("Ano")
    ax.plot(grupo["Ano"], grupo["IDH"], marker="o", markersize=3, linewidth=1.5, label=sigla)

ax.set_title("Evolução do IDH por estado (1991–2024)")
ax.set_xlabel("Ano")
ax.set_ylabel("IDH")
ax.set_ylim(0.3, 0.9)
ax.legend(ncol=3, bbox_to_anchor=(1.02, 1), loc="upper left", title="UF")
fig.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'